In [1]:
import ultralytics
from ultralytics import YOLO
from PIL import Image, ImageDraw, ImageFont
import requests

import os
import cv2

from matplotlib import patches, text, patheffects
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import shutil
import yaml

ultralytics.checks()

Ultralytics 8.4.63 🚀 Python-3.13.1 torch-2.6.0 CPU (Apple M2 Pro)
Setup complete ✅ (10 CPUs, 16.0 GB RAM, 305.7/460.4 GB disk)


In [2]:
#optional: a lot of warnings come up
import warnings
warnings.filterwarnings('ignore')

In [3]:
#if the prediction's (x1, y1) or (x2, y2) lands in the white portion of the filter, ignore it
#to help us out, a helper function that checks if true
def point_in_filter(x, y, filter):
    #check only the first slice, hence [0]
    if filter[int(x), filter[:, 0, 0].shape[0] - int(y)][0] == 255:
        return True
    else:
        return False

In [4]:
best_model = YOLO("best.pt")

In [5]:
base_dir = "/Users/jonathanzhu/nematostella_videos/imgs/"
img_folders = os.listdir(base_dir)
img_folders

['wt_25c_6_dpf_fert_04_10_2026',
 'rpa_bb_16C_9_dpf_celldish_04_02_2026',
 'rpa_bb_16C_3_dpf_movedtodish_03_27_2026',
 'rpa_aa_16C_3_dpf_fert_04_24_2026',
 'wt_25c_7_dpf_fert_04_10_2026',
 'wt_16C_2_dpf_03_26_2026',
 'rpa_bb_25C_8_dpf_celldish_04_01_2026',
 'rpa_aa_25C_3_dpf_03_27_2026_celldish',
 'rpa_bb_16C_3_dpf_fert_04_24_2026',
 'glw_ac_16c_3_dpf_fert_04_24_2026',
 'rpa_bb_16C_6_dpf_celldish_03_30_2026',
 'wt_25c_4_dpf_fert_04_10_2026',
 'wt_25C_6_dpf_celldish_03_30_2026',
 'rpa_aa_16C_3_dpf_movedtodish_03_27_2026',
 'wt_25c_5_dpf_fert_04_10_2026',
 'wt_16C_8_dpf_celldish_04_01_2026',
 'rpa_aa_16C_6_dpf_fert_04_24_2026',
 'wt_16C_3_dpf_03_27_2026_celldish',
 '.DS_Store',
 'rpa_bb_16C_2_dpf_03_26_2026',
 'rpa_aa_25C_8_dpf_celldish_04_01_2026',
 'rpa_aa_16C_7_dpf_fert_04_24_2026',
 'glw_ac_16c_5_dpf_fert_04_24_2026',
 'rpa_bb_25C_3_dpf_03_27_2026_celldish',
 'rpa_bb_16C_5_dpf_fert_04_24_2026',
 'rpa_aa_16C_6_dpf_celldish_03_30_2026',
 'rpa_bb_16C_6_dpf_fert_04_24_2026',
 'glw_ac_16c

In [6]:
current_dir = base_dir + img_folders[1] #folder with the images

images_list = os.listdir(current_dir)
images_list = [i for i in images_list if i != ".DS_Store"] #remove system file
images_list = [current_dir + "/" + i for i in images_list]

filter = cv2.imread("/Users/jonathanzhu/nematostella_videos/imgs/" + img_folders[1] + "/filter.png")

In [7]:
df_full = pd.DataFrame() #the dataframe that contains everything from this stack

for img in images_list:
    try: 
        results = best_model(img, verbose=False)
    except: 
        continue
    else:
        for r in results:
            labs = []
            b = r.boxes.data  # Boxes object for bounding box outputs
            df_res = pd.DataFrame(np.array(b))
            df_res.columns = ["x1", "y1", "x2", "y2", "conf", "class"]
            df_res.insert(0, "image_filename", img)

            for j in range(len(df_res)):
                if point_in_filter(df_res.iloc[j]["x1"], df_res.iloc[j]["y1"], filter):
                    labs.append(True)
                elif point_in_filter(df_res.iloc[j]["x2"], df_res.iloc[j]["y2"], filter):
                    labs.append(True)
                else:
                    labs.append(False)

            df_res["in_filter"] = labs
            df_res = df_res[df_res.in_filter == False]

            df_full = pd.concat([df_full, df_res], ignore_index=True)

df_full.to_csv(current_dir + "/positions.csv")
print("Saved to " + current_dir + "/positions.csv")

Saved to /Users/jonathanzhu/nematostella_videos/imgs/rpa_bb_16C_9_dpf_celldish_04_02_2026/positions.csv
